# NeuroLens Colab setup and data integration check

This notebook connects the Git repository to BraTS/BRISC stored in Drive, checks the saved split manifests, inspects BRISC metadata without preprocessing, then runs a single real BraTS batch through the model and losses. It does **not** start full training or generate metrics.

In [ ]:
# 1. Check the Colab GPU runtime before installing project dependencies.
import importlib.util, sys
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('numpy') is None:
    raise RuntimeError('Select a standard Colab GPU runtime. This notebook will not install or replace PyTorch/CUDA/NumPy.')
import torch, numpy as np
TORCH_VERSION, NUMPY_VERSION = torch.__version__, np.__version__
print('Torch:', TORCH_VERSION, '| NumPy:', NUMPY_VERSION)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > T4 GPU in Colab.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive and clone/pull the source repository.
from google.colab import drive
from pathlib import Path
import os, subprocess
drive.mount('/content/drive')
REPO_URL = os.environ.get('NEUROLENS_REPO_URL', '')  # Set to your GitHub HTTPS/SSH clone URL.
PROJECT_DIR = Path('/content/NeuroLens')
if not REPO_URL: raise ValueError('Set NEUROLENS_REPO_URL to the project GitHub repository URL.')
if (PROJECT_DIR / '.git').is_dir():
    origin = subprocess.check_output(['git','-C',str(PROJECT_DIR),'remote','get-url','origin'], text=True).strip()
    if origin != REPO_URL: raise RuntimeError(f'Existing checkout origin {origin!r} differs from REPO_URL; choose another PROJECT_DIR.')
    subprocess.check_call(['git','-C',str(PROJECT_DIR),'pull','--ff-only'])
else:
    subprocess.check_call(['git','clone',REPO_URL,str(PROJECT_DIR)])
%cd $PROJECT_DIR

In [ ]:
# 3. Install project dependencies. requirements.txt intentionally excludes Torch and NumPy.
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements.txt'])
import torch, numpy as np
if torch.__version__ != TORCH_VERSION or np.__version__ != NUMPY_VERSION:
    raise RuntimeError('Dependency resolution changed Torch or NumPy. Stop here and restore the Colab runtime versions before continuing.')
print('Core runtime versions preserved:', torch.__version__, np.__version__)
subprocess.check_call([sys.executable,'check_environment.py'])

In [ ]:
# 4. Resolve paths from config.yaml or environment variables.
# Set path variables in the Colab environment, or edit config.yaml in the cloned repo.
from neurolens.config import load_config
cfg = load_config('config.yaml')
raw_paths = {'BraTS root':cfg.brats_root or os.environ.get('NEUROLENS_BRATS_ROOT',''),
             'BRISC root':cfg.brisc_root or os.environ.get('NEUROLENS_BRISC_ROOT',''),
             'train manifest':cfg.manifest_train or os.environ.get('NEUROLENS_MANIFEST_TRAIN',''),
             'validation manifest':cfg.manifest_val or os.environ.get('NEUROLENS_MANIFEST_VAL',''),
             'test manifest':cfg.manifest_test or os.environ.get('NEUROLENS_MANIFEST_TEST','')}
for name, value in raw_paths.items():
    if not value or not Path(value).exists(): raise FileNotFoundError(f'{name} path missing or invalid: {value!r}. Configure YAML or NEUROLENS_* environment variables.')
BRATS_ROOT, BRISC_ROOT = Path(raw_paths['BraTS root']), Path(raw_paths['BRISC root'])
TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST = (Path(raw_paths[k]) for k in ('train manifest','validation manifest','test manifest'))
os.environ['NEUROLENS_BRATS_ROOT'] = str(BRATS_ROOT)
os.environ['NEUROLENS_BRISC_ROOT'] = str(BRISC_ROOT)
os.environ['NEUROLENS_MANIFEST_TRAIN'] = str(TRAIN_MANIFEST)
os.environ['NEUROLENS_MANIFEST_VAL'] = str(VAL_MANIFEST)
os.environ['NEUROLENS_MANIFEST_TEST'] = str(TEST_MANIFEST)
print('BraTS:', BRATS_ROOT, '| BRISC:', BRISC_ROOT)
print('Manifest paths validated; source manifest files are read-only.')

In [ ]:
# 5. Read-only dataset inspection: case discovery, manifest schemas, BRISC README/tree/checksums.
cmd = [sys.executable,'inspect_datasets.py','--brats-root',str(BRATS_ROOT),'--brisc-root',str(BRISC_ROOT),
       '--train-manifest',str(TRAIN_MANIFEST),'--val-manifest',str(VAL_MANIFEST),'--test-manifest',str(TEST_MANIFEST),
       '--id-column',cfg.manifest_id_column]
subprocess.check_call(cmd)
print('Review the BRISC README and manifest labels above before implementing or running its preprocessing.')

In [ ]:
# 6. One real paired BraTS batch: NIfTI read -> model forward -> all losses. No optimizer/training.
# Uses the saved train manifest, preserving its patient assignments.
subprocess.check_call([sys.executable,'integration_test.py','--config','config.yaml','--batch-size','2'])

## Ready for training setup
The integration check validates one real batch only. Once you have reviewed the BRISC manifest/label inspection and confirmed preprocessing, BRISC can be implemented as an independent classification/segmentation loader. Full NeuroLens training is intentionally a separate command and is not run by this setup notebook.